## What this notebook does

**GOAL: Extract datapoints and home locations of users in US during the period of interest.**

**WHEN: Use this notebook once per _period of interest (poi)_.**  
Outputs can be used for all regions of interest within the US in the chosen poi.

### Inputs
User data tables, from schema `paas_cda_pe_v3`: 
- table: **device_location_upleveled**
- table: **device_recurring_area**

### Outputs
- `datapoints/US_yyyy_mm.csv`, US level; all measured datapoints, aggregated per user in the US.
- `home_locations/US_yyyy_mm.csv`, US level; inferred home locations of users. 

## Settings

In [ ]:
path = '/home/jovyan/Main/mobility-bias_K'

In [ ]:
# max considered lag between event date and processing date in days
process_lag = 7

date_format = "%Y%m%d"

In [ ]:
# period of interest
# 2019 done in other notebook! (See intro)
# April 2019 - pre COVID
# start_date = '20190421'; end_date = '20190430' #a: 01-10, 0 lag #b: 11-20, 0 lag #c: 21-30 + lag = 21-07

# Okt 2019
# start_date = '20191001'; end_date = '20191031'


# April 2020 - during COVID lockdown/start lockdown
# start_date = '20200401'; end_date = '20200430'

# Okt 2020
# start_date = '20201001'; end_date = '20201031'


# April 2021 - late COVID
# start_date = '20210401'; end_date = '20210430'

# Okt 2021
# start_date = '20211001'; end_date = '20211031'


# April 2022 - post COVID
# start_date = '20220401'; end_date = '20220430'

# Okt 2022
start_date = '20221001'; end_date = '20221002'

In [ ]:
period = f'{start_date[:4]}_{start_date[4:6]}'

file_name = 'US_' + period
file_name_export = 'TEST_US_' + period

## Setup

In [ ]:
import os
os.chdir(path)

In [ ]:
# !pip install pyhive

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import time
from datetime import datetime, timedelta

from pyhive import trino

connection = trino.connect(
    host="localhost",
    port=9090,
    catalog="data_provider_name" # removed to ensure dataprovider anonymity
)

def read_sql(query:str) -> pd.DataFrame:
    return pd.read_sql(query, connection)

In [ ]:
# add process_lag days to given date
def lag_date(date, process_lag, date_format):
    return datetime.strftime(datetime.strptime(date, date_format) + timedelta(days=process_lag), date_format)

In [ ]:
first_date_to_compute = start_date
last_date_to_compute = lag_date(end_date, process_lag, date_format)

# slightly different period layout to match format event_zoned_datetime in table
period_event = f'{period[:4]}-{period[5:]}'

print(file_name)
print(first_date_to_compute, last_date_to_compute)

<a id='datapoints'></a>
## Extract aggregated number of datapoints per user
N.B: For 2019 (and earlier), time-outs may occur due to sheer size of data per month. If the case, extract per day/few days and combine later.

In [ ]:
# load the data schema for device location (datapoints)
schema_name = {'cda': 'data_provider_name.paas_cda_pe_v3'} # renoved dataprovider name
data_table = f"{schema_name['cda']}.device_location_uplevelled"

In [ ]:
%%time
# extract and save count of datapoints for all users with country_code 'US'
# ~1h (2020); 1h49 (2021); 1h25 (2022)
# 2019: US_2019_04a: 20190401 20190410 => 1h3
#       US_2019_04b: 20190411 20190420 => 1h6
#       US_2019_04c: 20190421 20190507 => 0h51
# US_2020_10: 1h54
# US_2021_10: 1h21

datapoints = read_sql(
    f"""
    select 
        user_id, 
        count(*) as data_points
    from {data_table}
    where
        processing_date between {first_date_to_compute} and {last_date_to_compute}
        and country_code = 'US'
        and substr(event_zoned_datetime,1,7) = '{period_event}'
    group by user_id
    """
)

datapoints.to_csv('datapoints/%s.csv' % file_name_export, index=False)

In [ ]:
display(datapoints)
print('Total data_points measured:', sum(datapoints['data_points']))

In [ ]:
# check aggregated datapoints distribution
# plt.hist(datapoints, bins=np.arange(0,100))
# plt.show()

---

<a id='homeloc'></a>
## Extract home locations
Works well for 2020, 2021 and 2022 data. Currently working on solution for 2019.

In [ ]:
# load the schema for home work locations
schema_name = {'cda': 'data_provider_name.paas_cda_pe_v3'} # renoved dataprovider name
hw_table = f"{schema_name['cda']}.device_recurring_area"

In [ ]:
%%timee
home_locations = read_sql(
    f"""
    select 
        user_id, 
        replace(substring(max_by(block_group_id, home_counts), 4, 15),'.','') as block_group_id
    from 
        (select 
            user_id, 
            block_group_id, 
            count(*) as home_counts
        from {hw_table}
        where 
            tag_type_code = 'HOME'
            and snapshot_event_date between {start_date} and {end_date}
            and country_code = 'US'
            and block_group_id is not null
        group by 
            user_id,
            block_group_id
        )
    group by user_id
    """
)

home_locations.to_csv('home-locations/' + file_name_export + '.csv', index=False)

In [ ]:
display(home_locations)